In [14]:
import numpy as np
from numpy.ma.core import zeros_like

A = np.array([[2, 4, 3], [-1, 0, 4], [3, 1, 2], [1, -3, 5], [1, 2, -2]])

v = 1/6*(A[:, 0] - np.array([-4, 0, 0, 0, 0]).T)
tau = 2/(np.dot(v, v))

print(A, v, tau)


[[ 2  4  3]
 [-1  0  4]
 [ 3  1  2]
 [ 1 -3  5]
 [ 1  2 -2]] [ 1.         -0.16666667  0.5         0.16666667  0.16666667] 1.5


np.float64(1.3333333333333333)

In [23]:
H = np.eye(5) - tau*np.outer(v, v)

In [24]:
H @ A

array([[-4.00000000e+00, -2.50000000e+00, -2.75000000e+00],
       [-4.16333634e-17,  1.08333333e+00,  4.95833333e+00],
       [ 0.00000000e+00, -2.25000000e+00, -8.75000000e-01],
       [ 7.63278329e-17, -4.08333333e+00,  4.04166667e+00],
       [ 4.16333634e-17,  9.16666667e-01, -2.95833333e+00]])

In [29]:
G = np.array([[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 3/5, 0, -4/5], [0, 0, 0, 1, 0], [0, 0, 4/5, 0, 3/5]])
G

array([[ 1. ,  0. ,  0. ,  0. ,  0. ],
       [ 0. ,  1. ,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0.6,  0. , -0.8],
       [ 0. ,  0. ,  0. ,  1. ,  0. ],
       [ 0. ,  0. ,  0.8,  0. ,  0.6]])

In [30]:
G @ np.array([7, -1, 3, 2, -4]).T

array([ 7.0000000e+00, -1.0000000e+00,  5.0000000e+00,  2.0000000e+00,
        4.4408921e-16])

In [33]:
M = np.array([[7, 0, 0, 0, 0], [-1, 4, 2, 0, 0], [3, 3, -6, 8, 0], [0, 1, 4, 1, 9], [0, 0, 0, 2, -5], [0, 0, 0, 6, 2], [0, 0, 0, 0, 4]])
M

array([[ 7,  0,  0,  0,  0],
       [-1,  4,  2,  0,  0],
       [ 3,  3, -6,  8,  0],
       [ 0,  1,  4,  1,  9],
       [ 0,  0,  0,  2, -5],
       [ 0,  0,  0,  6,  2],
       [ 0,  0,  0,  0,  4]])

In [36]:
rows, cols = M.shape
N = np.zeros_like(M)

for r in range(rows):
    for j in range(cols):
        if M[r][j] != 0:
            N[3+r-j][j] = M[r][j]

N

array([[ 0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0],
       [ 0,  0,  2,  8,  9],
       [ 7,  4, -6,  1, -5],
       [-1,  3,  4,  2,  2],
       [ 3,  1,  0,  6,  4],
       [ 0,  0,  0,  0,  0]])

In [37]:
import numpy as np

M = np.array([
    [7, 0, 0, 0, 0],
    [-1, 4, 2, 0, 0],
    [3, 3, -6, 8, 0],
    [0, 1, 4, 1, 9],
    [0, 0, 0, 2, -5],
    [0, 0, 0, 6, 2],
    [0, 0, 0, 0, 4]
])

rows, cols = M.shape

# 1. Automatically detect bandwidth
ku = 0
kl = 0
for r in range(rows):
    for j in range(cols):
        if M[r, j] != 0:
            if j > r: ku = max(ku, j - r)
            if r > j: kl = max(kl, r - j)

# 2. Create the storage matrix N (height = kl + ku + 1)
N = np.zeros((kl + ku + 1, cols))

# 3. Fill N using the 0-indexed LAPACK formula: N[ku + i - j, j]
for r in range(rows):
    for j in range(cols):
        if M[r, j] != 0:
            row_in_N = ku + r - j
            N[row_in_N, j] = M[r, j]

print("LAPACK Band Storage Matrix N:")
print(N)

LAPACK Band Storage Matrix N:
[[ 0.  0.  2.  8.  9.]
 [ 7.  4. -6.  1. -5.]
 [-1.  3.  4.  2.  2.]
 [ 3.  1.  0.  6.  4.]]


In [40]:
A = np.array([[3, -1, 0, 0, 5, 0, 2], [0, 0, 6, 1, 0, 8, 0], [3, 0, 2, 0, 0, 0, -7], [0, 2, -3, 0, 1, 0, 0], [0, 0, 0, 4, 0, 0, 1]])
A

array([[ 3, -1,  0,  0,  5,  0,  2],
       [ 0,  0,  6,  1,  0,  8,  0],
       [ 3,  0,  2,  0,  0,  0, -7],
       [ 0,  2, -3,  0,  1,  0,  0],
       [ 0,  0,  0,  4,  0,  0,  1]])

In [41]:
from scipy.sparse import csc_matrix
A = csc_matrix(A)
A

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 15 stored elements and shape (5, 7)>

In [43]:
print("Data:   ", A.data)
print("Indices:", A.indices)
print("Indptr: ", A.indptr)

Data:    [ 3  3 -1  2  6  2 -3  1  4  5  1  8  2 -7  1]
Indices: [0 2 0 3 1 2 3 1 4 0 3 1 0 2 4]
Indptr:  [ 0  2  4  7  9 11 12 15]


In [44]:
in_v = np.array([7, -1, 3, 2, -4]).T
out_v = np.array([7, -1, 5, 2, 0]).T

In [59]:
A = np.array([[2, 4, 3], [-1, 0, 4], [3, 1, 2], [1, -3, 5], [1, 2, -2]])
A
tau = 3/2
v = np.array([1, -1/6, 1/2, 1/6, 1/6]).T

In [58]:
tau*v.T@A

array([6.  , 6.5 , 5.75])

In [61]:
A - np.outer(v,tau*v.T@A
 )

array([[-4.        , -2.5       , -2.75      ],
       [ 0.        ,  1.08333333,  4.95833333],
       [ 0.        , -2.25      , -0.875     ],
       [ 0.        , -4.08333333,  4.04166667],
       [ 0.        ,  0.91666667, -2.95833333]])